# 51 — Rubber projectiles in the blowdown pipe: ball, bore sweep, cylinder + head shapes

The rig of notebook 50: a 1 L container at **50 bar** (absolute), 20 °C, a straight pipe with a valve 20 mm from the
container that opens at t = 0. Every projectile is **solid rubber (1100 kg/m³)**, a tight fit in the bore, its rear
5 mm after the valve.

1. **The ball** (Ø 10 mm, 0.58 g): coupled CFD. The gas drives the ball, and the ball (a moving seal, a piston)
   shapes the flow: the driver gas expands behind it, a shock runs ahead of it.
2. **Bore sweep**: 7, 10, 12, 15 mm, the ball always the bore's size → ball exit speed vs pipe diameter.
3. **Shapes**: a cylinder with a **hemispherical** or a **paraboloidal** head, L/D = 1 … 10 in the 10 mm bore → which
   leaves with the most momentum?
4. **No ball**: r–x Mach maps of the plain blowdown at a series of times.

**How the CFD works** (`sonic_ball.py`): OpenFOAM v2412 `rhoCentralFoam` on a moving mesh. The projectile is a rigid
6-DoF body (axial only, `sixDoFRigidBodyDisplacement`) moved by the pressure and viscous forces on its faces. The bore
cells stretch behind it and squeeze ahead of it (`displacementLayeredMotion`). The run stops as it reaches the exit.
The CFD body slides without friction. Rubber friction (μ × contact pressure × contact area) is added on top, using
the CFD's gas force along the bore. A **0-D gun model** (lumped driver gas, piston-driven shock ahead) runs alongside.

In [ ]:
DEBUG = False                                  # real OpenFOAM (no mocks)
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)
RUN_CFD = True                                 # solves what is missing; finished cases are reused
import json, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
import sonic_flow as sf, sonic_ball as sb, sonic_flow_report as rep
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.color": "#e4e3df", "axes.spines.top": False,
                     "axes.spines.right": False, "lines.linewidth": 2.0, "legend.frameon": False})
BLUE, ORANGE, AQUA, VIOLET, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#52514e"
rig, ball, cond = sf.Rig(), sb.Ball(), sb.BALL_CONDITIONS
OUT = Path("output/sonic_ball"); RES = Path("sonic_ball_results"); RES.mkdir(exist_ok=True)
def save(fig, name):
    fig.savefig(RES / f"{name}.png", dpi=140, bbox_inches="tight")

## 1. The ball
### Design

In [ ]:
print(f"ball: Ø {ball.diameter} mm, {ball.density:.0f} kg/m³, mass {ball.mass*1e3:.3f} g, "
      f"rubber friction {ball.friction:.1f} N (μ {ball.mu}, contact {ball.contact_pressure/1e6:.2f} MPa on a {ball.sphere_band} mm band)")
print(f"rear at x = {ball.rear_x} mm (valve {rig.valve_x} mm); the rear runs {rig.pipe_length - ball.rear_x - ball.diameter:.0f} mm to the muzzle")
F0 = (cond.p0 - cond.p_amb) * rig.exit_area_m2
print(f"full container pressure on the ball: {F0:.0f} N -> {F0/ball.mass/9.81:,.0f} g")
cad = sb.make_cad(RES / "cad", rig, ball)
from vegeta import dedalus
fig = dedalus.plot_views(dedalus.load_step(RES / "cad/ball_rig.step"))
fig.suptitle("Supply rig with the 10 mm rubber ball in the bore"); plt.show()

In [ ]:
mesh = sf.Mesh()                               # 0.5 mm in the bore
fig, ax = plt.subplots(figsize=(11, 3.4))
colors = {"container": "#b0aea6", "pipeBehindBall": BLUE, "pipeAheadOfBall": ORANGE, "ambient": "#d8d6cf"}
for x0, x1, r0, r1, nx, nr, ex, er, zone in sb.blocks(rig, mesh, ball):
    ax.add_patch(plt.Rectangle((x0, r0), x1 - x0, r1 - r0, color=colors[zone], alpha=0.55))
    if zone.startswith("pipe"):
        ax.text((x0 + x1) / 2, r1 + 0.6, f"{nx}×{nr}", ha="center", fontsize=7)
ax.add_patch(plt.Circle((ball.centre(rig), 0), ball.diameter / 2, color="k", alpha=0.8))
ax.axvline(rig.valve_x, color="k", ls="--", lw=1); ax.text(rig.valve_x + 0.5, 9, "valve", fontsize=8)
ax.set_xlim(-20, 130); ax.set_ylim(0, 12); ax.set_aspect("equal"); ax.set_xlabel("x [mm]"); ax.set_ylabel("r [mm]")
ax.set_title("Moving zones: blue stretches behind the ball, orange squeezes ahead; grey stands still", loc="left", fontsize=10)
plt.show()

### Coupled CFD: air on two meshes (0.5 mm and 1 mm, a mesh check), CO₂ on 1 mm

In [ ]:
coarse = sb.bore_setup(10.0)                    # (rig, ball, mesh) with 1 mm cells
RUNS = {
    "Air, 0.5 mm": (sb.solve(OUT / "air", sf.AIR, rig, cond, mesh, ball, run=RUN_CFD), BLUE, "-"),
    "Air, 1 mm":   (sb.solve(OUT / "sweep_air_d10", sf.AIR, coarse[0], cond, coarse[2], coarse[1], run=RUN_CFD, chunk=50e-6), BLUE, "--"),
    "CO2, 1 mm":   (sb.solve(OUT / "co2_d10_coarse", sf.CO2, coarse[0], cond, coarse[2], coarse[1], run=RUN_CFD, chunk=50e-6), ORANGE, "-"),
}
H = {k: sb.histories(c) for k, (c, _, _) in RUNS.items()}
MZ = {k: sb.muzzle(sb.ball_track(c), ball, rig) for k, (c, _, _) in RUNS.items()}
GUN = {"Air": sb.gun(sf.AIR, rig, ball=ball, friction=0.0), "CO2": sb.gun(sf.CO2, rig, ball=ball, friction=0.0)}
for k, m in MZ.items():
    h = H[k]
    print(f"{k:12s}: leaves after {m['time']*1e3:.3f} ms at {m['speed']:.1f} m/s (frictionless); "
          f"peak force {h['ball_force'].max():.0f} N, peak rear pressure {h['p_rear'].max()/1e5:.1f} bar")
for g, r in GUN.items():
    print(f"0-D {g:4s}   : leaves after {r['t'][-1]*1e3:.3f} ms at {r['u'][-1]:.1f} m/s")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for k, (c, col, ls) in RUNS.items():
    h = H[k]; t = h["ball_t"] * 1e3
    F = np.interp(h["ball_t"], h["force_t"], h["ball_force"])
    axes[0][0].plot(t, h["ball_x"] * 1e3 + ball.diameter / 2, color=col, ls=ls, label=f"{k} CFD")
    axes[0][1].plot(t, h["ball_u"], color=col, ls=ls, label=f"{k} CFD")
    axes[1][0].plot(t, F / ball.mass / 9.81 / 1e3, color=col, ls=ls, label=f"{k} CFD")
    axes[1][1].plot(t, F, color=col, ls=ls, label=f"{k} CFD")
for g, r in GUN.items():
    col = BLUE if g == "Air" else ORANGE
    for ax, y in zip(axes.flat, (r["ball_x"] * 1e3 + ball.diameter / 2, r["u"], r["a"] / 9.81 / 1e3, r["force"])):
        ax.plot(r["t"] * 1e3, y, color=col, ls=":", lw=1.4, label=f"{g} 0-D gun model")
axes[0][0].axhline(rig.pipe_length, color=MUTED, ls="-.", lw=1); axes[0][0].text(0.01, rig.pipe_length + 1, "pipe exit", fontsize=8)
for ax, (title, unit) in zip(axes.flat, (("Ball front position", "mm"), ("Ball velocity", "m/s"),
                                          ("Ball acceleration", "1000 g"), ("Net gas force on the ball", "N"))):
    ax.set_title(title, loc="left"); ax.set_ylabel(unit)
for ax in axes[1]:
    ax.set_xlabel("time after the valve opens [ms]")
axes[0][1].legend(fontsize=8); fig.tight_layout(); save(fig, "ball_motion"); plt.show()

### The flow around the ball, with and without the ball

In [ ]:
plain = {}
for g in ("air", "co2"):
    for p in (Path("output/sonic_flow") / f"{g}_startup", Path("output/sonic_flow") / g, Path("output/sonic_smoke") / g):
        if (p / "postProcessing/exitPlane").exists():
            plain["Air" if g == "air" else "CO2"] = sf.histories(p); break
t_end = max(h["ball_t"][-1] for h in H.values()) * 1e3
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
for k, (c, col, ls) in RUNS.items():
    h = H[k]
    axes[0][0].plot(h["pball_t"] * 1e3, h["p_rear"] / 1e5, color=col, ls=ls, label=f"{k}: behind")
    axes[0][0].plot(h["pball_t"] * 1e3, h["p_front"] / 1e5, color=col, ls=ls, lw=1, alpha=0.7, label=f"{k}: ahead")
    axes[0][1].plot(h["t"] * 1e3, h["inlet_mdot"], color=col, ls=ls, label=f"{k} with ball")
    axes[1][0].plot(h["t"] * 1e3, h["exit_U_area"], color=col, ls=ls, label=f"{k} with ball")
    axes[1][1].plot(h["t"] * 1e3, h["exit_mdot"], color=col, ls=ls, label=f"{k} with ball")
for g, q in plain.items():
    k = q["t"] * 1e3 <= t_end; col = BLUE if g == "Air" else ORANGE
    axes[0][1].plot(q["t"][k] * 1e3, q["valve_mdot"][k], color=col, ls=":", label=f"{g} no ball")
    axes[1][0].plot(q["t"][k] * 1e3, q["exit_U_area"][k], color=col, ls=":", label=f"{g} no ball")
    axes[1][1].plot(q["t"][k] * 1e3, q["exit_mdot"][k], color=col, ls=":", label=f"{g} no ball")
for ax, (title, unit) in zip(axes.flat, (("Pressure on the ball's faces", "bar"), ("Mass flow out of the container", "kg/s"),
                                          ("Exit gas velocity (area-averaged)", "m/s"), ("Mass flow at the pipe exit", "kg/s"))):
    ax.set_title(title, loc="left"); ax.set_ylabel(unit); ax.legend(fontsize=7)
for ax in axes[1]:
    ax.set_xlabel("time after the valve opens [ms]")
fig.tight_layout(); save(fig, "ball_flow"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
for ax, k in zip(axes, ("Air, 0.5 mm", "CO2, 1 mm")):
    c = RUNS[k][0]
    times, rows = sb.axis_profiles(c, fields=("p",))
    xs = np.linspace(-0.02, 0.15, 400); P = np.full((len(times), len(xs)), np.nan); tr = sb.ball_track(c)
    for i, (t, r) in enumerate(zip(times, rows)):
        P[i] = np.interp(xs, r["x"], r["p"]); xc = np.interp(t, tr["t"], tr["x"])
        P[i, np.abs(xs - xc) < ball.diameter * 0.5e-3] = np.nan
    im = ax.pcolormesh(xs * 1e3, times * 1e3, P / 1e5, cmap="viridis", shading="nearest", vmin=0, vmax=50)
    for s in (-0.5, 0.5):
        ax.plot((tr["x"] + s * ball.diameter * 1e-3) * 1e3, tr["t"] * 1e3, color="w", lw=1.2)
    ax.axvline(rig.valve_x, color="w", ls=":", lw=0.8); ax.axvline(rig.pipe_length, color="w", ls="--", lw=0.8)
    ax.set_title(f"{k}: pressure on the axis (white: the ball)", loc="left"); ax.set_xlabel("x [mm]")
axes[0].set_ylabel("time [ms]"); fig.colorbar(im, ax=axes, label="p [bar]", shrink=0.85); save(fig, "ball_wave_diagram"); plt.show()

## 2. Bore sweep: ball exit speed vs pipe diameter (ball = bore, air, 5 cells across the radius)

In [ ]:
SW = sb.sweep(sf.AIR, run=RUN_CFD)
D = np.array([r["diameter_mm"] for r in SW])
gun_d = {fr: [sb.gun(sf.AIR, sb.bore_setup(d)[0], ball=sb.bore_setup(d)[1], friction=fr) for d in D] for fr in (0.0, None)}
cfd_fr = []
for d, r in zip(D, SW):
    rg, b, _ = sb.bore_setup(d)
    x, F = sb.force_vs_travel(OUT / f"sweep_air_d{d:g}")
    cfd_fr.append(sb.exit_with_friction(x, F, b.mass, b.friction, (rg.pipe_length - b.rear_x - b.diameter) * 1e-3))
print(f"{'D [mm]':>7s} {'mass [g]':>9s} {'CFD [m/s]':>10s} {'CFD+friction':>13s} {'0-D [m/s]':>10s} {'exit [ms]':>10s} {'momentum [g m/s]':>17s}")
for d, r, vf, g0 in zip(D, SW, cfd_fr, gun_d[0.0]):
    print(f"{d:7.0f} {r['ball_mass_g']:9.3f} {r['speed']:10.1f} {vf:13.1f} {g0['u'][-1]:10.1f} {r['time']*1e3:10.3f} {r['ball_mass_g']*vf:17.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(D, [r["speed"] for r in SW], color=BLUE, marker="s", ms=8, label="CFD, frictionless")
axes[0].plot(D, cfd_fr, color=BLUE, marker="o", ms=6, ls="--", label="CFD gas force + rubber friction")
axes[0].plot(D, [g["u"][-1] for g in gun_d[0.0]], color=MUTED, marker=".", ls=":", lw=1.4, label="0-D gun model, frictionless")
for d, v in zip(D, cfd_fr):
    axes[0].annotate(f"{v:.0f}", (d, v), textcoords="offset points", xytext=(6, -12), fontsize=8)
axes[0].set_title("Ball exit speed vs pipe diameter", loc="left"); axes[0].set_ylabel("m/s"); axes[0].set_ylim(bottom=0)
axes[1].plot(D, [r["ball_mass_g"] * v for r, v in zip(SW, cfd_fr)], color=BLUE, marker="o", label="momentum [g·m/s]")
axes[1].plot(D, [0.5 * r["ball_mass_g"] * 1e-3 * v * v for r, v in zip(SW, cfd_fr)], color=ORANGE, marker="s", label="kinetic energy [J]")
axes[1].set_title("What the ball carries out (CFD + friction)", loc="left")
for ax in axes:
    ax.set_xlabel("pipe = ball diameter [mm]"); ax.set_xticks(D); ax.legend(fontsize=8)
fig.tight_layout(); save(fig, "ball_speed_vs_diameter"); plt.show()

## 3. Shapes: cylinder + hemispherical or paraboloidal head, L/D = 1 … 10 (10 mm bore, air)

While a body seals the bore, the gas sees only its rear face and its mass, so the coupled CFD runs once per mass (the
paraboloid series). The hemisphere-headed bodies use the gas force interpolated between the two CFD runs that bracket
their mass. Each body's rear runs the same 65 mm to the muzzle: the barrel is lengthened in front for the long ones,
and the CFD keeps one diameter of slug. The head changes the mass and the length of cylinder pressed on the wall, which
sets the friction. Its aerodynamic advantage only counts in flight, after the muzzle.

Friction is the uncertain input. The contact pressure of a rubber body squeezed into a bore depends on the
interference and the rubber, and it grows when 50 bar squashes the body axially. So it is shown at 0.05, 0.15
(reference: ~10 N on the 10 mm ball) and 0.3 MPa.

In [ ]:
bodies = {f"{h}_ld{ld}": sb.projectile(h, ld) for h in ("hemisphere", "paraboloid") for ld in (1, 3, 10)}
sb.make_projectile_cad(RES / "cad", bodies)
fig, axes = plt.subplots(2, 1, figsize=(12, 4.6), sharex=True)
for ax, head in zip(axes, ("hemisphere", "paraboloid")):
    y0 = 0
    for ld in sb.LD_VALUES:
        b = sb.projectile(head, ld); pts = np.array(sb.profile(b))
        ax.fill(np.r_[pts[:, 0], pts[::-1, 0]], np.r_[pts[:, 1], -pts[::-1, 1]] + y0, color=BLUE if head == "hemisphere" else ORANGE, alpha=0.8)
        ax.text(b.L + 2, y0, f"L/D {ld}: {b.mass*1e3:.2f} g", va="center", fontsize=7)
        y0 -= 12
    ax.set_title(f"cylinder + {head} head", loc="left"); ax.set_aspect("equal"); ax.axis("off")
plt.show()

In [ ]:
SH = sb.shape_sweep(sf.AIR, run=RUN_CFD)
travel = 0.065
PC = (0.05e6, 0.15e6, 0.30e6)
res = {}
for head in ("paraboloid", "hemisphere"):
    for pc in (0.0,) + PC:
        rows = []
        for ld in sb.LD_VALUES:
            b = sb.projectile(head, ld, contact_pressure=max(pc, 1.0))
            x, F = sb.force_for_mass(SH, b.mass) if head == "hemisphere" else (SH["x"][SH["ld"].index(ld)], SH["F"][SH["ld"].index(ld)])
            v = sb.exit_with_friction(x, F, b.mass, b.friction if pc > 0 else 0.0, travel)
            rows.append(dict(ld=ld, mass=b.mass, friction=b.friction if pc > 0 else 0.0, v=v, p=b.mass * v, E=0.5 * b.mass * v * v))
        res[(head, pc)] = rows
print("frictionless CFD exit speeds (paraboloid series):", [f"{v:.0f}" for v in SH["speed"]], "m/s")
for (head, pc), rows in res.items():
    best = max(rows, key=lambda r: r["p"])
    print(f"{head:10s} contact {pc/1e6:4.2f} MPa: best momentum at L/D {best['ld']:>2}: {best['p']*1e3:6.1f} g·m/s "
          f"({best['mass']*1e3:.2f} g at {best['v']:.0f} m/s, friction {best['friction']:.0f} N)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ls_pc = {0.0: "-", 0.05e6: "--", 0.15e6: "-", 0.30e6: ":"}
for (head, pc), rows in res.items():
    col = BLUE if head == "hemisphere" else ORANGE
    lw = 1.2 if pc == 0.0 else 2.0
    alpha = 0.45 if pc == 0.0 else 1.0
    lab = f"{head}, " + ("frictionless" if pc == 0 else f"contact {pc/1e6:.2f} MPa")
    L = [r["ld"] for r in rows]
    axes[0].plot(L, [r["p"] * 1e3 for r in rows], color=col, ls=ls_pc[pc], lw=lw, alpha=alpha, marker="o", ms=4, label=lab)
    axes[1].plot(L, [r["v"] for r in rows], color=col, ls=ls_pc[pc], lw=lw, alpha=alpha, marker="o", ms=4)
    axes[2].plot(L, [r["E"] for r in rows], color=col, ls=ls_pc[pc], lw=lw, alpha=alpha, marker="o", ms=4)
    if pc == 0.15e6:
        b = max(rows, key=lambda r: r["p"]); axes[0].plot(b["ld"], b["p"] * 1e3, marker="*", ms=16, color=col, mec="k")
axes[1].plot(SH["ld"], SH["speed"], color=MUTED, marker="s", ls="none", ms=6, label="CFD runs (paraboloid, frictionless)")
for ax, (title, unit) in zip(axes, (("Exit momentum (★: best at 0.15 MPa)", "g·m/s"), ("Exit speed", "m/s"), ("Exit kinetic energy", "J"))):
    ax.set_title(title, loc="left"); ax.set_ylabel(unit); ax.set_xlabel("L / D"); ax.set_xticks(sb.LD_VALUES)
axes[0].legend(fontsize=7.5); axes[1].legend(fontsize=7.5)
fig.tight_layout(); save(fig, "shape_momentum"); plt.show()

## 4. No ball: r–x Mach maps at different times

In [ ]:
def plain_case(g, early):
    p = Path("output/sonic_flow") / (g + ("_startup" if early else ""))
    return p if rep._times(p) else Path("output/sonic_smoke") / g
early = {"Air": plain_case("air", True), "CO$_2$": plain_case("co2", True)}
fig = rep.mach_maps(early, (0.1, 0.15, 0.2, 0.3, 0.45, 0.6), title="Start-up: the valve opens at t = 0 (no ball)")
save(fig, "mach_maps_startup"); plt.show()
late = {"Air": plain_case("air", False), "CO$_2$": plain_case("co2", False)}
fig = rep.mach_maps(late, (1.0, 2.0, 5.0, 10.0), title="The established jet (no ball)")
save(fig, "mach_maps_late"); plt.show()
print("early from", {k: str(v) for k, v in early.items()}, "\nlate from", {k: str(v) for k, v in late.items()})

## 5. Reading it

*(filled in from the results)*